In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

Построение графа

In [2]:
def make_graph(
    n_nodes=12,
    n_groups=3,
    min_neighbors=2,
    max_neighbors=4,
    within_group_bias=3.0,
    weight_range=(0.2, 1.0),
    rng=None,
):
    """
    Create a weighted directed graph.

    A[i, j] represents the influence of node j on node i.
    The diagonal is zero, and each row sums to 1.
    Groups have equal sizes and contain consecutive node indices.
    """
    integer_params = (n_nodes, n_groups, min_neighbors, max_neighbors)
    for value in integer_params:
        if isinstance(value, (bool, np.bool_)) or not isinstance(
            value, (int, np.integer)
        ):
            raise ValueError("Node, group, and neighbor counts must be integers.")

    if n_nodes < 2 or not 1 <= n_groups <= n_nodes:
        raise ValueError("Require n_nodes >= 2 and 1 <= n_groups <= n_nodes.")

    if n_nodes % n_groups != 0:
        raise ValueError("n_nodes must be divisible by n_groups.")

    if not 1 <= min_neighbors <= max_neighbors < n_nodes:
        raise ValueError(
            "Require 1 <= min_neighbors <= max_neighbors < n_nodes."
        )

    if not np.isfinite(within_group_bias) or within_group_bias <= 0:
        raise ValueError("within_group_bias must be positive and finite.")

    low, high = weight_range
    if not (np.isfinite([low, high]).all() and 0 < low <= high):
        raise ValueError("Weight bounds must be finite and satisfy 0 < low <= high.")

    if rng is None:
        rng = np.random.default_rng()
    elif not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be a np.random.Generator or None.")

    groups = np.repeat(np.arange(n_groups), n_nodes // n_groups)
    nodes = np.arange(n_nodes)
    A = np.zeros((n_nodes, n_nodes), dtype=float)

    for i in range(n_nodes):
        # Exclude the receiving node from its possible sources.
        candidates = nodes[nodes != i]
        k = int(rng.integers(min_neighbors, max_neighbors + 1))

        # Assign higher sampling weights to candidates in the same group.
        probabilities = np.where(
            groups[candidates] == groups[i],
            within_group_bias,
            1.0,
        )
        probabilities /= probabilities.sum()

        sources = rng.choice(
            candidates,
            size=k,
            replace=False,
            p=probabilities,
        )

        # Normalize incoming edge weights so that each row sums to one.
        weights = rng.uniform(low, high, size=k)
        A[i, sources] = weights / weights.sum()

    return A

In [3]:
import csv
from pathlib import Path

import numpy as np


def next_step(x, A, rng, a=0.4, b=0.3, noise_std=0.1):
    """Update node values using a fixed graph and additive process noise."""
    x = np.asarray(x, dtype=float)
    noise = rng.normal(0.0, noise_std, size=x.shape)

    return a * x + b * (A @ x) + noise


def make_dataset_csv(
    path,
    n_steps=6000,
    burn_in=200,
    a=0.4,
    b=0.3,
    noise_std=0.1,
    seed=42,
    graph_params=None,
):
    """
    Create one graph and write simulated states to CSV row by row.

    The CSV contains the initial state after burn-in, followed by
    n_steps transitions. Return the graph used to generate the data.
    """
    for name, value, minimum in (
        ("n_steps", n_steps, 1),
        ("burn_in", burn_in, 0),
    ):
        if (
            isinstance(value, (bool, np.bool_))
            or not isinstance(value, (int, np.integer))
            or value < minimum
        ):
            raise ValueError(f"{name} must be an integer >= {minimum}.")

    if not np.isfinite([a, b, noise_std]).all() or noise_std < 0:
        raise ValueError(
            "a and b must be finite; "
            "noise_std must be finite and nonnegative."
        )

    if abs(a) + abs(b) >= 1:
        raise ValueError("Require abs(a) + abs(b) < 1 for this generator.")

    graph_params = {} if graph_params is None else dict(graph_params)

    if "rng" in graph_params:
        raise ValueError("Set randomness through seed, not graph_params['rng'].")

    # Separate graph randomness from process noise.
    graph_seed, noise_seed = np.random.SeedSequence(seed).spawn(2)
    graph_rng = np.random.default_rng(graph_seed)
    noise_rng = np.random.default_rng(noise_seed)

    # Create the graph once and keep it fixed.
    A = make_graph(rng=graph_rng, **graph_params)
    x = np.zeros(A.shape[0])

    # Discard the initial transient.
    for _ in range(burn_in):
        x = next_step(x, A, noise_rng, a=a, b=b, noise_std=noise_std)

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)

        writer.writerow(["step", *[f"node_{i}" for i in range(len(x))]])
        writer.writerow([0, *x])

        # Generate and immediately write each subsequent state.
        for t in range(1, n_steps + 1):
            x = next_step(x, A, noise_rng, a=a, b=b, noise_std=noise_std)
            writer.writerow([t, *x])

    return A

In [4]:
A = make_dataset_csv(
    path="../data/processed/synthetic_static.csv",
    n_steps=6000,
    burn_in=200,
    noise_std=0.1,
    seed=42,
    graph_params={
        "n_nodes": 12,
        "n_groups": 3,
        "min_neighbors": 2,
        "max_neighbors": 4,
    },
)

# Preserve the true graph for diagnostics.
np.savetxt(
    "../data/processed/synthetic_static_graph.csv",
    A,
    delimiter=",",
)